In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import numpy as np
import torch
import torch.nn as nn
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.resnet50 import preprocess_input
from transformers import BertTokenizer, BertModel

# =========================
# PATHS
# =========================
PROJECT_DIR = "/content/drive/MyDrive/final-multimodal-project"

FUSION_MODEL_PATH = os.path.join(PROJECT_DIR, "models", "fusion_best_model.keras")
TEXT_MODEL_PATH   = os.path.join(PROJECT_DIR, "models", "text_only_model.keras")
IMAGE_MODEL_PATH  = os.path.join(PROJECT_DIR, "models", "image_auth_model_torch.pth")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# LOAD TEXT MODELS
# =========================
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert = BertModel.from_pretrained("bert-base-uncased").to(DEVICE)
bert.eval()

text_only_model = load_model(TEXT_MODEL_PATH)
fusion_model = load_model(FUSION_MODEL_PATH)

print("✅ Text & Fusion models loaded")

# =========================
# LOAD IMAGE MODEL (PyTorch)
# =========================
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 28 * 28, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

image_model = SimpleCNN().to(DEVICE)
image_model.load_state_dict(torch.load(IMAGE_MODEL_PATH, map_location=DEVICE))
image_model.eval()

print("✅ Image authenticity model loaded")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

✅ Text & Fusion models loaded
✅ Image authenticity model loaded


In [3]:
def predict_text_only(text):
    with torch.no_grad():
        tokens = tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors="pt"
        ).to(DEVICE)

        output = bert(**tokens)
        text_emb = output.last_hidden_state[:, 0, :].cpu().numpy()

    prob = text_only_model.predict(text_emb, verbose=0)[0][0]

    return {
        "prediction": "Fake News" if prob >= 0.5 else "Real News",
        "confidence": f"{prob*100:.2f}%" if prob >= 0.5 else f"{(1-prob)*100:.2f}%"
    }


In [4]:
from torchvision import transforms
from PIL import Image

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

def predict_image_only(image_path):
    img = Image.open(image_path).convert("RGB")
    img = transform(img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        prob = torch.sigmoid(image_model(img)).item()

    return {
        "image_prediction": "AI-generated" if prob >= 0.5 else "Authentic",
        "confidence": f"{prob*100:.2f}%" if prob >= 0.5 else f"{(1-prob)*100:.2f}%"
    }


In [ ]:
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input

mobilenet = MobileNetV2(
    weights="imagenet",
    include_top=False,
    pooling="avg"
)

print("✅ MobileNetV2 loaded for fusion inference")


In [5]:
def predict_text_and_image(text, image_path):
    # ---- Text embedding ----
    with torch.no_grad():
        tokens = tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors="pt"
        ).to(DEVICE)

        output = bert(**tokens)
        text_emb = output.last_hidden_state[:, 0, :].cpu().numpy()

    # ---- Image embedding (ResNet-style) ----
    img = image.load_img(image_path, target_size=(224, 224))
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)

    img_emb = tf.keras.applications.ResNet50(
        weights="imagenet",
        include_top=False,
        pooling="avg"
    ).predict(x, verbose=0)

    # ---- Fusion ----
    fusion_input = np.concatenate([text_emb, img_emb], axis=1)
    prob = fusion_model.predict(fusion_input, verbose=0)[0][0]

    return {
        "news_prediction": "Fake News" if prob >= 0.5 else "Real News",
        "confidence": f"{prob*100:.2f}%" if prob >= 0.5 else f"{(1-prob)*100:.2f}%"
    }


In [6]:
sample_text = "Bangladesh unrest updates: Bangladesh summons Indian envoy over security concerns for Missions"
sample_image = "/content/test.jpg"

print("\nTEXT ONLY:")
print(predict_text_only(sample_text))

print("\nIMAGE ONLY:")
print(predict_image_only(sample_image))

print("\nTEXT + IMAGE:")
print(predict_text_and_image(sample_text, sample_image))



TEXT ONLY:
{'prediction': 'Fake News', 'confidence': '88.55%'}

IMAGE ONLY:
{'image_prediction': 'AI-generated', 'confidence': '57.35%'}

TEXT + IMAGE:
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


ValueError: Input 0 of layer "multimodal_fusion" is incompatible with the layer: expected shape=(None, 2048), found shape=(1, 2816)